In [1]:
from datasets import load_dataset

# Stream the full English MIRACL corpus directly from the jsonl.gz shards on main.
# (load_dataset("miracl/miracl-corpus", ...) fails: the repo has a legacy loading
# script, which datasets >= 4 no longer supports.)
ds = load_dataset(
    "json",
    data_files="hf://datasets/miracl/miracl-corpus/miracl-corpus-v1.0-en/docs-*.jsonl.gz",
    split="train",
    streaming=True,
)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
it = iter(ds)

first = next(it)
second = next(it)

print(first)
print(second)

{'docid': '12#0', 'title': 'Anarchism', 'text': 'Anarchism is a political philosophy that advocates self-governed societies based on voluntary, cooperative institutions and the rejection of hierarchies those societies view as unjust. These institutions are often described as stateless societies, although several authors have defined them more specifically as institutions based on non-hierarchical or free associations. Anarchism holds capitalism, the state, and representative democracy to be undesirable, unnecessary, and harmful.'}
{'docid': '12#1', 'title': 'Anarchism', 'text': 'While opposition to the state is central, many forms of anarchism specifically entail opposing authority or hierarchical organisation in the conduct of all human relations. Anarchism is often considered a far-left ideology, and much of anarchist economics and anarchist legal philosophy reflect anti-authoritarian interpretations of communism, collectivism, syndicalism, mutualism, or participatory economics.'}


In [8]:
# Queries + relevance judgments live in the separate miracl/miracl repo, as plain TSVs.
# Splits with qrels: train (~2863 queries), dev (799). test-a/test-b topics have no public qrels.
MIRACL_EN = "hf://datasets/miracl/miracl/miracl-v1.0-en"

topics = load_dataset(
    "csv",
    data_files=f"{MIRACL_EN}/topics/topics.miracl-v1.0-en-dev.tsv",
    delimiter="\t",
    column_names=["query_id", "query"],
    split="train",
    streaming=True,
)


it = iter(topics)

first = next(it)
second = next(it)

print(first)
print(second)

{'query_id': 0, 'query': 'Is Creole a pidgin of French?'}
{'query_id': 20, 'query': "What percentage of the Earth's atmosphere is oxygen?"}


In [9]:
i = 0
while i < 10: 
     next_element = next(it)
     print(next_element)
     i += 1

{'query_id': 32, 'query': 'When did Marxism develop?'}
{'query_id': 33, 'query': "Which Assassin's Creed is the latest released?"}
{'query_id': 34, 'query': 'Why is it called guerrilla?'}
{'query_id': 35, 'query': 'What was the first film directed by Andrei Tarkovsky?'}
{'query_id': 37, 'query': 'Who wrote the song "Happy Days"?'}
{'query_id': 43, 'query': 'What was the first Atlantic hurricane in 2000?'}
{'query_id': 44, 'query': 'What is the main version of Islam practiced in Iraq?'}
{'query_id': 47, 'query': 'When did Aristagoras become leader of Miletus?'}
{'query_id': 49, 'query': "What is Captain Cold's real name?"}
{'query_id': 61, 'query': 'Where are polycyclic aromatic hydrocarbons found?'}


In [13]:
from query_taxonomy.features import CorpusIdentifierExtractor

# fresh full iteration of the streaming dev split (~799 queries)
queries = [row["query"] for row in topics]

extractor = CorpusIdentifierExtractor()
corpus_ids = extractor.extract(queries)

with_ids = [q for q in corpus_ids.queries if q.spans]
print(f"queries with >=1 identifier: {len(with_ids)}/{len(queries)}")
for type_, doc in corpus_ids.documents.items():
    print(f"{type_.value:18} docs={len(doc.spans):4} diversity={doc.diversity:4}")

queries with >=1 identifier: 23/799
number             docs=  21 diversity=  17
http_status_code   docs=   2 diversity=   1


In [14]:
# which queries got tagged, per type
for type_, doc in corpus_ids.documents.items():
    print(f"--- {type_.value}")
    for doc_id, matches in list(doc.spans.items())[:5]:
        print(f"  {[m.text for m in matches]}  <-  {queries[int(doc_id)]!r}")

--- number
  ['2000']  <-  'What was the first Atlantic hurricane in 2000?'
  ['2015']  <-  'Where was the DreamHack Open Cluj-Napoca 2015 held?'
  ['5']  <-  'Who discovered 5-Hydroxyeicosatetraenoic acid?'
  ['2']  <-  'When was USS Lexington (CV-2) built?'
  ['16']  <-  'Can you vote at 16 years in Argentina?'
--- http_status_code
  ['470 Fire']  <-  'When did the Number 470 Fire Bell become heritage-listed?'
  ['470 Fire']  <-  'Who created the Number 470 Fire Bell?'


In [12]:
from collections import Counter

# most common surface forms per type (corpus-level dfs)
for type_, doc in corpus_ids.documents.items():
    print(type_.value, Counter(doc.dfs).most_common(5))

number [('2', 3), ('2000', 2), ('2017', 2), ('2015', 1), ('5', 1)]
http_status_code [('470 Fire', 2)]
